# 08 - GitHub Actions CI/CD for Azure ML

In this session I set up a CI/CD pipeline that automatically submits a training job to Azure ML whenever I push changes to my training scripts on GitHub. This means I don't have to manually run notebooks every time I update my code, GitHub handles it automatically.

## What I built

### Service Principal
I created a service principal in Azure — essentially a non-personal account that GitHub Actions can use to authenticate with Azure. I generated the credentials using the Azure CLI and stored them as a GitHub secret called `AZURE_CREDENTIALS`.

### GitHub Actions Workflow
I created `.github/workflows/train.yml` which defines the automated pipeline. It triggers whenever I push changes to the `pipeline_scripts/` folder. When triggered it:
- Spins up a fresh Ubuntu machine
- Logs into Azure using the service principal credentials
- Installs the required Python packages
- Runs `.github/scripts/submit_training.py` which connects to my Azure ML workspace and submits a training job

### Compute Cluster
I created `ml-learning-cluster` — a compute cluster rather than a compute instance. I had to switch to a cluster because compute instances are personal and the service principal couldn't submit jobs to them. The cluster is set to 0 minimum nodes so it costs nothing when idle.

## How the trigger works

The workflow only fires when I change files inside `pipeline_scripts/`. This is intentional — I only want to retrain when the actual training code changes, not when I update notebooks or documentation.

To trigger it manually I add a comment to `train.py` and push:

```bash
git add .
git commit -m "update training script"
git push
```

GitHub picks up the push, detects a change in `pipeline_scripts/`, and runs the workflow automatically.

## Issues I ran into

**First attempt failed** — the workflow tried to submit a job to `ml-compute-md` (my compute instance) but the service principal doesn't have permission to use personal compute instances. Fixed by creating `ml-learning-cluster` and updating the compute target.

**Workflow not triggering** — pushing changes to `.github/` files doesn't trigger the workflow because the path filter only watches `pipeline_scripts/**`. Had to push a change to `train.py` to trigger it.

**Second attempt succeeded** — workflow completed in 1m 19s, job submitted to Azure ML cluster successfully.

In [5]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Connected:", ml_client.workspaces.get("ml-learning-workspace").name)

# Show recent jobs including the CI/CD triggered run
jobs = list(ml_client.jobs.list(max_results=5))
for job in jobs:
    print(f"Job: {job.display_name} | Status: {job.status} | Experiment: {job.experiment_name}")

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Connected: ml-learning-workspace
Job: CI-CD Training Run | Status: Failed | Experiment: azure-ml-learning
Job: Test Credit Risk Environment | Status: Completed | Experiment: environment-test
Job: nice_floor_h82zbhl3 | Status: Completed | Experiment: prepare_image
Job: Iris Training Pipeline | Status: Completed | Experiment: iris-pipeline
Job: crimson_bag_2p392qxt8b | Status: Completed | Experiment: iris-automl
Job: modest_date_6nhwz4bgyz | Status: Failed | Experiment: iris-automl
